# Clonal evolution of the go-or-grow switch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sisyga/biolgca/blob/aidevelop/docs/source/zoo/clonal_go_or_grow.ipynb)

*A new model, the research model `go_or_grow_glioblastoma` of earlier
biolgca versions. No paper: the questions and runs below are a starting
point for projects.*

**Question.** In glioblastoma, cells seem to either migrate or divide, not
both at once (the *go-or-grow* hypothesis), and the tumour consists of
clones that differ in how fast they divide. Suppose that a driver mutation
changes both a cell's division rate and its switch between migrating and
dividing. Which clones win in a growing tumour, and where? Does the switch
evolve along with the division rate?

In [ ]:
# In Google Colab this cell installs BioLGCA (about a minute); elsewhere it does nothing.
import importlib.util
import subprocess
import sys

if "google.colab" in sys.modules and importlib.util.find_spec("lgca") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "biolgca @ git+https://github.com/sisyga/biolgca@aidevelop"], check=True)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import lgca
from lgca import zoo
from lgca.model import run_model
from lgca.zoo import clonal_go_or_grow as clonal
from lgca.zoo._clones import family_trait, largest_clones

zoo.parameter_table("clonal_go_or_grow")

## The model

Identity-based cells without volume exclusion on a hexagonal lattice: a
node holds any number of cells, and division slows down as it fills up to
its carrying capacity $K$. Resting cells (in the rest channel) have the
proliferative phenotype, moving cells the migratory one. Every cell carries
two traits, its birth rate $r_b$ and its switch parameter $\kappa$. In every
time step:

1. **Switch:** every cell rests with probability
   $$r_\kappa(\rho_N) = \tfrac12\left(1 + \tanh\big(\kappa(\rho_N - \theta)\big)\right)$$
   and moves otherwise, where $\rho_N$ is the number of cells on its node
   and the neighbouring nodes over $(b + 1)K$.
2. **Death:** every cell dies with probability $r_d$.
3. **Division:** resting cells divide with probability $r_b(1 - n/K)$. A
   daughter acquires a *driver mutation* with probability $r_m$: it founds
   a new clone, its birth rate is multiplied by $f = 1.1$, and its $\kappa$
   changes by a normal step of standard deviation $\Delta\kappa$.
4. **Migration:** moving cells pick a random direction and move.

$\kappa > 0$ is the *attractive* strategy (rest where it is crowded, move
where it is sparse), $\kappa < 0$ the *repulsive* one (move away from
crowding), $\kappa = 0$ the *independent* one (rest with probability 1/2).
The tumour starts from $K$ resting cells at the centre, all of one clone
with $\kappa_0 = 0$. To separate the two traits, the same model runs with
$\kappa$ fixed (`evolve_kappa=False`): drivers then change only the birth
rate.

In [ ]:
for operator in clonal.build_spec().dynamics.operators:
    print(operator)

runs = {evolve: run_model(clonal.build_spec(evolve_kappa=evolve), showprogress=False) for evolve in (True, False)}

## Clones, switch and birth rate across the tumour

After 400 steps on a 150 × 150 lattice (250 × 250 and 700 steps with
`build_spec(full=True)`), per node: the clone that holds most of its cells
(the nine clones that dominate most nodes in colour, all others in light grey),
the mean $\kappa$ and the mean birth rate:

In [ ]:
from matplotlib.colors import ListedColormap, Normalize

clone_colours = ListedColormap(["0.8"] + list(plt.get_cmap("tab10").colors[:9]))  # others in light grey
fig, axes = plt.subplots(2, 3, figsize=(15, 9.5), constrained_layout=True)
for row, (evolve, run) in enumerate(runs.items()):
    lattice = run.lgca
    maps = clonal.node_maps(lattice)
    empty = maps["cells"] == 0
    panels = (("clone", largest_clones(maps["clone"]), {"cmap": clone_colours, "vmin": -0.5, "vmax": 9.5,
                                                          "cbarlabel": "rank of the clone (0: others)"}),
              ("mean κ", maps["kappa"], {"cmap": "RdBu_r", "vmin": -2, "vmax": 2, "cbarlabel": "κ"}),
              ("mean birth rate", maps["r_b"], {"cmap": "viridis", "vmin": 0.2, "vmax": 0.3, "cbarlabel": "$r_b$"}))
    for column, (title, values, options) in enumerate(panels):
        axis = axes[row, column]
        lattice.plot_scalarfield(values, ax=axis, tight_layout=False, mask=empty, **options)
        x, y = lattice.xcoords[~empty], lattice.ycoords[~empty]
        axis.set(xlim=(x.min() - 2, x.max() + 2), ylim=(y.min() - 2, y.max() + 2))
        axis.set_title(f"κ {'evolves' if evolve else 'fixed'}: {title}")
plt.show()
plt.close(fig)

for evolve, run in runs.items():
    summary = clonal.core_and_rim(run.lgca)
    kappa, r_b = summary["kappa"], summary["r_b"]
    print(f"κ {'evolves' if evolve else 'fixed  '}: {summary['cells']:7d} cells in {summary['clones']} clones; "
          f"mean κ core {kappa['core']:+.2f}, rim {kappa['rim']:+.2f}; "
          f"mean r_b core {r_b['core']:.3f}, rim {r_b['rim']:.3f}")

What the maps show:

- **Clones form sectors.** A driver that arises near the edge spreads with
  the edge and claims a wedge of the tumour; one that arises in the core
  stays a patch.
- **The switch evolves towards repulsion.** Mean $\kappa$ turns negative
  across the tumour (core is the inner half of the cells by distance from
  the centre, rim the outermost tenth). Repulsive cells move away from
  crowding, so they reach the empty space at the edge first, where they
  divide freely; their clones take over the edge and, as it moves on, the
  whole tumour.
- **Faster division is selected** in both runs: the mean birth rate rises
  above its initial 0.2, a little more when the switch evolves too, since
  clones at the edge divide more and so acquire more drivers.

How the clones rose and fell over time, in Muller plots. Each band is a
clone, drawn inside the clone it descends from; clones that never held 1 %
of the tumour are left out. With $\kappa$ evolving, the bands are coloured
by the clone's $\kappa$:

In [ ]:
for evolve, run in runs.items():
    lattice = run.lgca
    if evolve:
        lattice.family_props["kappa"] = family_trait(lattice, "kappa")
        options = {"prop": "kappa", "cmap": "RdBu_r", "norm": Normalize(-2, 2), "label_map": lambda family: None}
    else:
        options = {}
    fig, axis, *_ = lattice.muller_plot(cutoff_rel=0.01, **options)
    axis.set_title(f"κ {'evolves' if evolve else 'fixed'}")
    fig.set_size_inches(10, 3.5)
    plt.show()
    plt.close(fig)

## Does the evolving switch make the tumour grow faster?

The population over time for three seeds each, with $\kappa$ evolving and
fixed (about 40 s):

In [ ]:
populations = {(evolve, 1): run.data["population"] for evolve, run in runs.items()}  # seed 1 ran above
for evolve in (True, False):
    for seed in (2, 3):
        run = run_model(clonal.build_spec(evolve_kappa=evolve, seed=seed), showprogress=False)
        populations[evolve, seed] = run.data["population"]
steps = run.data.steps("population")

fig, axis = plt.subplots(figsize=(7, 4))
for (evolve, seed), population in populations.items():
    axis.plot(steps, population, color="tab:red" if evolve else "tab:blue",
              label=("κ evolves" if evolve else "κ fixed") if seed == 1 else None)
axis.set(xlabel="time step $k$", ylabel="cells", yscale="log")
axis.legend()
plt.show()
plt.close(fig)

for seed in (1, 2, 3):
    ratio = populations[True, seed][-1] / populations[False, seed][-1]
    print(f"seed {seed}: the tumour with an evolving switch has {ratio:.2f} times the cells after {steps[-1]} steps")

The tumour with an evolving switch grows faster, by a fifth to a half in
cells after 400 steps: repulsive clones carry the edge outwards. This is a growth
*rate*: once a tumour fills the lattice, both end at the same size.

## Notes on the model

- **Initial switch.** The tumour starts with $\kappa_0 = 0$. With
  $\kappa_0 = 5$, the default of the earlier research model, cells in the
  small initial colony almost all move (their neighbourhood is far below
  $\theta = 0.5$), few divide, and the colony shrinks: an Allee effect, as
  in the zoo entry *Allee effect*. Starting from independent switching
  gives evolution room to move in both directions.
- **Order of the rules.** Switch, death, division, migration, as in the
  research model `go_or_grow_glioblastoma`; the test suite checks that the
  two give the same runs.
- **Size of the mutations.** A driver changes $\kappa$ by a normal step
  with $\Delta\kappa = 1$. The research model used 0.2; then $\kappa$
  moves so slowly that within 400 steps the two tumours hardly differ (by
  3–6 % in cells, $\kappa$ around $-0.1$). Over longer runs the same
  picture emerges.
- **Only drivers.** Every mutation raises the birth rate by 10 %; there are
  no deleterious or neutral mutations (the zoo entry *Drivers, passengers
  and mutational meltdown* has both).

## Explore it

A smaller tumour with the mean $\kappa$ per node as the view. Change the
mutation probability $r_m$ and the death rate $r_d$ while it grows:

In [ ]:
explorer = lgca.explore(
    clonal.build_spec(size=70),
    {"dynamics.operators[go_or_grow.growth].mutation.probability": (0.0, 0.05),  # r_m
     "death_rate": (0.0, 0.15)},  # r_d
    view="mean kappa",
    measure=["population", "kappa", "r_b"],
    steps_per_frame=2,
)
explorer

## Things to try

1. **Death.** Raise $r_d$. In the zoo entry *Evolution of phenotypic
   plasticity*, death decides whether attractive or repulsive switches win.
   Does it here, with clones instead of a continuously mutating $\kappa$?
2. **The threshold.** Change $\theta$. With a low threshold the core is
   crowded above it; which $\kappa$ wins there?
3. **Allee effect.** Start with $\kappa_0 = 5$ (`build_spec(kappa=5)`) and
   find the smallest initial colony (`capacity`, or more cells at the
   centre) that survives. Can mutations rescue a colony that would die out?
4. **Where are the drivers?** Record the position of every mutation (write
   an observer that reads `lgca.maxfamily` each step, or compare
   `clonal.node_maps` over time). Do the clones that win arise at the rim?